# Preparação do TPDD_Honglin_CLS

**Projeto:** SafraCerta  
**Disciplina:** Deep Learning

## Objetivo

Preparar o dataset TPDD_Honglin_CLS para treinamento de um modelo
YOLO de classificação de doenças em folhas de tabaco.

O notebook realiza:

- download do dataset;
- identificação das classes;
- análise dos splits;
- contagem das imagens;
- organização para YOLO Classification;
- verificação de arquivos inválidos;
- detecção de duplicatas;
- geração de relatório do dataset.

In [ ]:
!pip install -q datasets pillow imagehash pandas matplotlib

In [ ]:
from datasets import load_dataset

DATASET_NAME = "TamAko783/TPDD_Honglin_CLS"

dataset = load_dataset(DATASET_NAME)

print(dataset)

In [ ]:
print(dataset["train"].features)

print("\nPrimeiro registro:")
print(dataset["train"][0])

In [ ]:
classes = dataset["train"].features["label"].names

print(f"Total de classes: {len(classes)}\n")

for id_classe, classe in enumerate(classes):
    print(f"{id_classe:2} → {classe}")

In [ ]:
total = 0

for split in dataset.keys():

    quantidade = len(dataset[split])

    print(f"{split:12}: {quantidade}")

    total += quantidade

print("-" * 30)
print(f"Total: {total}")

In [ ]:
import pandas as pd
from collections import Counter

dados = []

for split in dataset.keys():

    contagem = Counter(dataset[split]["label"])

    for label_id, quantidade in contagem.items():

        dados.append({
            "classe": classes[label_id],
            "split": split,
            "quantidade": quantidade
        })

df = pd.DataFrame(dados)

tabela = df.pivot(
    index="classe",
    columns="split",
    values="quantidade"
).fillna(0).astype(int)

display(tabela)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(14, 7))

for i, ax in enumerate(axes.flat):

    exemplo = dataset["train"][i]

    ax.imshow(exemplo["image"])
    ax.set_title(classes[exemplo["label"]])
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path

OUTPUT = Path("/content/TPDD_Honglin_CLS_YOLO")

OUTPUT.mkdir(
    parents=True,
    exist_ok=True
)

print("Destino:", OUTPUT)

In [ ]:
splits = {
    "train": "train",
    "validation": "val",
    "test": "test"
}

for split_hf, split_yolo in splits.items():

    for classe in classes:

        pasta = OUTPUT / split_yolo / classe

        pasta.mkdir(
            parents=True,
            exist_ok=True
        )

print("Pastas criadas.")

In [ ]:
for split_hf, split_yolo in splits.items():

    print(f"Exportando {split_hf}...")

    for i, exemplo in enumerate(dataset[split_hf]):

        imagem = exemplo["image"]
        label = exemplo["label"]

        classe = classes[label]

        if imagem.mode != "RGB":
            imagem = imagem.convert("RGB")

        destino = (
            OUTPUT
            / split_yolo
            / classe
            / f"{split_hf}_{i:05d}.jpg"
        )

        imagem.save(destino, quality=95)

print("\nExportação concluída.")

In [ ]:
for split in ["train", "val", "test"]:

    total_split = sum(
        1 for arquivo in (OUTPUT / split).rglob("*.jpg")
    )

    print(f"{split}: {total_split}")

In [ ]:
from PIL import Image

invalidas = []

for arquivo in OUTPUT.rglob("*.jpg"):

    try:

        with Image.open(arquivo) as img:
            img.verify()

    except Exception as erro:

        invalidas.append({
            "arquivo": str(arquivo),
            "erro": str(erro)
        })

print("Imagens inválidas:", len(invalidas))

In [ ]:
import hashlib
from collections import defaultdict

def calcular_sha256(caminho):

    hash_sha = hashlib.sha256()

    with open(caminho, "rb") as arquivo:

        for bloco in iter(
            lambda: arquivo.read(8192),
            b""
        ):
            hash_sha.update(bloco)

    return hash_sha.hexdigest()


hashes = defaultdict(list)

for split in ["train", "val", "test"]:

    for arquivo in (OUTPUT / split).rglob("*.jpg"):

        hashes[calcular_sha256(arquivo)].append({
            "split": split,
            "arquivo": str(arquivo)
        })


duplicatas = {
    hash_imagem: arquivos
    for hash_imagem, arquivos in hashes.items()
    if len(arquivos) > 1
}

print(
    "Grupos de duplicatas exatas:",
    len(duplicatas)
)

In [ ]:
duplicatas_entre_splits = []

for arquivos in duplicatas.values():

    splits_encontrados = {
        item["split"]
        for item in arquivos
    }

    if len(splits_encontrados) > 1:
        duplicatas_entre_splits.append(arquivos)

print(
    "Duplicatas entre splits:",
    len(duplicatas_entre_splits)
)

In [ ]:
import imagehash

phashes = defaultdict(list)

for split in ["train", "val", "test"]:

    for arquivo in (OUTPUT / split).rglob("*.jpg"):

        with Image.open(arquivo) as img:

            phash = str(
                imagehash.phash(
                    img.convert("RGB")
                )
            )

        phashes[phash].append({
            "split": split,
            "arquivo": str(arquivo)
        })


duplicatas_visuais = []

for arquivos in phashes.values():

    splits_encontrados = {
        item["split"]
        for item in arquivos
    }

    if (
        len(arquivos) > 1
        and len(splits_encontrados) > 1
    ):
        duplicatas_visuais.append(arquivos)


print(
    "Possíveis duplicatas visuais entre splits:",
    len(duplicatas_visuais)
)

In [ ]:
train_total = sum(
    1 for x in (OUTPUT / "train").rglob("*.jpg")
)

val_total = sum(
    1 for x in (OUTPUT / "val").rglob("*.jpg")
)

test_total = sum(
    1 for x in (OUTPUT / "test").rglob("*.jpg")
)

print("=" * 60)
print("RELATÓRIO TPDD_Honglin_CLS")
print("=" * 60)

print(f"""
Classes: {len(classes)}

Train: {train_total}
Val:   {val_total}
Test:  {test_total}

Total: {train_total + val_total + test_total}

Imagens inválidas:
{len(invalidas)}

Grupos de duplicatas exatas:
{len(duplicatas)}

Duplicatas exatas entre splits:
{len(duplicatas_entre_splits)}

Possíveis duplicatas visuais entre splits:
{len(duplicatas_visuais)}
""")